In [1]:
import os
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)



In [2]:
possible_paths = [
    "/kaggle/input/sample_submission.csv",
    "/kaggle/input/stanford-covid-vaccine/sample_submission.csv",
    "/kaggle/data/sample_submission.csv",
    "/kaggle/data/stanford-covid-vaccine/sample_submission.csv",
]

sample_path = None
for p in possible_paths:
    if os.path.exists(p):
        sample_path = p
        break

if sample_path is None:
    raise FileNotFoundError(
        "Could not find sample_submission.csv in expected Kaggle locations. "
        f"Tried: {possible_paths}"
    )

df = pd.read_csv(sample_path)
df.head()



,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.0,0.0,0.0,0.0,0.0
1,id_00b436dec_1,0.0,0.0,0.0,0.0,0.0
2,id_00b436dec_2,0.0,0.0,0.0,0.0,0.0
3,id_00b436dec_3,0.0,0.0,0.0,0.0,0.0
4,id_00b436dec_4,0.0,0.0,0.0,0.0,0.0


In [3]:
train_paths = [
    "/kaggle/input/train.json",
    "/kaggle/input/stanford-covid-vaccine/train.json",
    "/kaggle/data/train.json",
    "/kaggle/data/stanford-covid-vaccine/train.json",
]
train_path = None
for p in train_paths:
    if os.path.exists(p):
        train_path = p
        break
if train_path is None:
    raise FileNotFoundError(
        "Could not find train.json in expected Kaggle locations. "
        f"Tried: {train_paths}"
    )

train = pd.read_json(train_path, lines=True)

targets = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]
seq_scored = int(train["seq_scored"].iloc[0])  # 68 in this competition
seq_length = int(train["seq_length"].iloc[0])  # 107 in this competition

if "SN_filter" in train.columns:
    train_hq = train[train["SN_filter"] == 1].copy()
    if "signal_to_noise" in train_hq.columns:
        SNR_FLOOR = 1.0
        train_hq = train_hq[train_hq["signal_to_noise"] >= SNR_FLOOR].copy()

    if len(train_hq) < 200:
        train_hq = train[train["SN_filter"] == 1].copy()
        if len(train_hq) < 100:
            train_hq = train
else:
    train_hq = train

if "signal_to_noise" in train_hq.columns:
    sn = train_hq["signal_to_noise"].to_numpy(dtype=np.float32)
    sn = np.nan_to_num(
        sn, nan=np.float32(0.0), posinf=np.float32(0.0), neginf=np.float32(0.0)
    )
    sn = np.clip(sn, np.float32(0.0), np.float32(10.0))
    sample_w = (sn + np.float32(1e-3)).astype(np.float32)  # (n,)
else:
    sample_w = np.ones(len(train_hq), dtype=np.float32)


def _trimmed_mean_per_pos(y_2d: np.ndarray, trim_q: float = 0.06) -> np.ndarray:
    """
    y_2d: (n, 68) float32 with possible NaNs.
    Returns per-position trimmed mean (68,) float32.
    """
    y = y_2d.astype(np.float32, copy=False)
    lo = np.nanquantile(y, trim_q, axis=0).astype(np.float32)
    hi = np.nanquantile(y, 1.0 - trim_q, axis=0).astype(np.float32)
    keep = (y >= lo[None, :]) & (y <= hi[None, :]) & np.isfinite(y)
    y_kept = np.where(keep, y, np.nan)
    m = np.nanmean(y_kept, axis=0).astype(np.float32)
    m_fallback = np.nanmean(y, axis=0).astype(np.float32)
    m = np.where(np.isfinite(m), m, m_fallback).astype(np.float32)
    return m


def _weighted_mean_per_pos(y_2d: np.ndarray, w_1d: np.ndarray) -> np.ndarray:
    """
    Minimal scoring-relevant change: replace weighted median with a stable weighted mean.
    This stays in the same "per-position prior" family but tends to reduce RMSE vs step-like medians.
    y_2d: (n, 68) float32 (may include NaNs/infs)
    w_1d: (n,) float32 nonnegative sample weights
    Returns per-position weighted mean (68,) float32.
    """
    y = y_2d.astype(np.float32, copy=False)
    w = w_1d.astype(np.float32, copy=False)

    mask = np.isfinite(y) & np.isfinite(w)[:, None] & (w[:, None] > 0)
    y0 = np.where(mask, y, np.float32(0.0)).astype(np.float32)
    w0 = np.where(mask, w[:, None], np.float32(0.0)).astype(np.float32)

    num = np.sum(y0.astype(np.float64) * w0.astype(np.float64), axis=0)
    den = np.sum(w0.astype(np.float64), axis=0) + 1e-12
    m = (num / den).astype(np.float32)

    # Fallback for any weird degenerate columns
    m_fallback = np.nanmean(np.where(np.isfinite(y), y, np.nan), axis=0).astype(
        np.float32
    )
    m = np.where(np.isfinite(m), m, m_fallback).astype(np.float32)
    m = np.where(np.isfinite(m), m, np.float32(0.0)).astype(np.float32)
    return m


pos_means = {}
SHRINK_ALPHA = np.float32(0.08)

TRIM_Q = 0.06

winsor_bounds = {}

for t in targets:
    y_list = train_hq[t].values
    y = np.stack(y_list).astype(np.float32)  # (n, 68)

    m_u_robust = _trimmed_mean_per_pos(y, trim_q=TRIM_Q)

    lo = np.nanquantile(y, TRIM_Q, axis=0).astype(np.float32)
    hi = np.nanquantile(y, 1.0 - TRIM_Q, axis=0).astype(np.float32)
    winsor_bounds[t] = (lo, hi)

    y_wins = np.clip(y, lo[None, :], hi[None, :]).astype(np.float32)

    # Scoring-relevant tweak: robust weighted mean (on winsorized values) instead of weighted median.
    m_w = _weighted_mean_per_pos(y_wins, sample_w)

    if np.any(~np.isfinite(m_w)):
        m_w = np.where(np.isfinite(m_w), m_w, m_u_robust).astype(np.float32)

    m = (np.float32(1.0) - SHRINK_ALPHA) * m_w + SHRINK_ALPHA * m_u_robust
    pos_means[t] = m.astype(np.float32)

pos_full = {}
for t in targets:
    full = np.empty(seq_length, dtype=np.float32)
    full[:seq_scored] = pos_means[t]
    full[seq_scored:] = pos_means[t][-1]
    pos_full[t] = full



In [4]:
scored_targets = ["reactivity", "deg_Mg_pH10", "deg_Mg_50C"]

best_scales = {}
w_flat = np.repeat(sample_w.astype(np.float64), seq_scored)  # (n*68,)

for t in scored_targets:
    y_raw = np.stack(train_hq[t].values).astype(np.float32)  # (n,68)

    lo = np.nanquantile(y_raw, TRIM_Q, axis=0).astype(np.float32)
    hi = np.nanquantile(y_raw, 1.0 - TRIM_Q, axis=0).astype(np.float32)

    y = np.clip(y_raw, lo[None, :], hi[None, :]).astype(np.float32)
    y = np.nan_to_num(
        y, nan=np.float32(0.0), posinf=np.float32(0.0), neginf=np.float32(0.0)
    )

    b = np.repeat(pos_means[t][None, :], repeats=y.shape[0], axis=0).astype(np.float32)

    y_flat = y.reshape(-1).astype(np.float64)
    b_flat = b.reshape(-1).astype(np.float64)

    den = float(np.dot(w_flat * b_flat, b_flat)) + 1e-12
    num = float(np.dot(w_flat * y_flat, b_flat))
    z = num / den  # z = 1/s

    if (not np.isfinite(z)) or z == 0.0:
        s_opt = 1.15
    else:
        s_opt = float(1.0 / z)

    s_opt = float(np.clip(s_opt, 0.9, 1.4))
    best_scales[t] = np.float32(s_opt)

SCALE = float(np.mean([best_scales[t] for t in scored_targets]))


def mcrmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    mse = np.mean((y_true - y_pred) ** 2, axis=0)  # (68,3)
    rmse_per_target = np.sqrt(np.mean(mse, axis=0))  # (3,)
    return float(np.mean(rmse_per_target))


Y_true_list = []
for t in scored_targets:
    y_raw = np.stack(train_hq[t].values).astype(np.float32)
    lo = np.nanquantile(y_raw, TRIM_Q, axis=0).astype(np.float32)
    hi = np.nanquantile(y_raw, 1.0 - TRIM_Q, axis=0).astype(np.float32)
    y = np.clip(y_raw, lo[None, :], hi[None, :]).astype(np.float32)
    y = np.nan_to_num(
        y, nan=np.float32(0.0), posinf=np.float32(0.0), neginf=np.float32(0.0)
    )
    Y_true_list.append(y)
Y_true = np.stack(Y_true_list, axis=-1)  # (n,68,3)

Y_pred_base = np.stack(
    [pos_means[t].astype(np.float32) for t in scored_targets], axis=-1
)[None, :, :]
Y_pred_base = np.repeat(Y_pred_base, repeats=Y_true.shape[0], axis=0)

Y_pred_scaled = Y_pred_base.copy().astype(np.float32)
for k, t in enumerate(scored_targets):
    Y_pred_scaled[:, :, k] = Y_pred_scaled[:, :, k] / best_scales[t]

cv_score = mcrmse(Y_true, Y_pred_scaled)
print(
    "Chosen per-target SCALE via weighted closed-form LS (scored targets only): "
    + ", ".join([f"{t}={float(best_scales[t]):.4f}" for t in scored_targets])
    + f" (mean={SCALE:.4f}; train MCRMSE={cv_score:.6f})"
)



Chosen per-target SCALE via weighted closed-form LS (scored targets only): reactivity=0.9944, deg_Mg_pH10=0.9952, deg_Mg_50C=0.9961 (mean=0.9952; train MCRMSE=0.331611)


In [5]:
required_cols = [
    "id_seqpos",
    "reactivity",
    "deg_Mg_pH10",
    "deg_pH10",
    "deg_Mg_50C",
    "deg_50C",
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns in sample submission template: {missing}"
    )

df = df[required_cols].copy()

seqpos = df["id_seqpos"].str.rsplit("_", n=1).str[-1].astype(int).to_numpy()
for t in targets:
    df[t] = pos_full[t][seqpos]



In [6]:
per_col_scale = {
    "reactivity": best_scales["reactivity"],
    "deg_Mg_pH10": best_scales["deg_Mg_pH10"],
    "deg_Mg_50C": best_scales["deg_Mg_50C"],
    "deg_pH10": np.float32(1.0),
    "deg_50C": np.float32(1.0),
}
for c in required_cols[1:]:
    df[c] = df[c].astype(np.float32) / per_col_scale[c]



In [7]:
if df.isna().any().any():
    raise ValueError("Submission contains NaNs after processing.")

for c in required_cols[1:]:
    df[c] = df[c].astype(np.float32)

df.head()



,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.588198,0.636635,2.065608,0.512055,0.673712
1,id_00b436dec_1,1.618903,2.239997,3.124010,2.211886,1.966280
2,id_00b436dec_2,1.241516,0.824685,1.014176,1.093297,1.130589
3,id_00b436dec_3,0.921827,0.643359,0.761141,0.895387,0.897201
4,id_00b436dec_4,0.723345,0.779316,0.804642,0.914681,0.803917


In [8]:
out_path = "submission.csv"
df.to_csv(out_path, index=False)

print(f"Wrote {out_path} with shape {df.shape} and columns {list(df.columns)}")
print(df.head(3))

Wrote submission.csv with shape (25680, 6) and columns ['id_seqpos', 'reactivity', 'deg_Mg_pH10', 'deg_pH10', 'deg_Mg_50C', 'deg_50C']
        id_seqpos  reactivity  deg_Mg_pH10  deg_pH10  deg_Mg_50C   deg_50C
0  id_00b436dec_0    0.588198     0.636635  2.065608    0.512055  0.673712
1  id_00b436dec_1    1.618903     2.239997  3.124010    2.211886  1.966280
2  id_00b436dec_2    1.241516     0.824685  1.014176    1.093297  1.130589
